# Vegetation Temporal Dynamics from Sentinel-1 Coherence Time Series
### Case study: agricultural parcels in Flevoland, The Netherlands

This notebook implements **use case 2**: using a **season-long coherence time series**
(not a full interferometric network inversion) to detect crop-growth and harvest dynamics,
via [`sentinel1_sar_coherence`](https://algorithm-catalogue.apex.esa.int/apps/sentinel1_sar_coherence)
on the Copernicus Data Space Ecosystem (CDSE) openEO back-end.

**Why coherence for vegetation?** Interferometric coherence over vegetated surfaces drops
as canopy structure changes between the two acquisitions of a pair (wind-driven motion,
growth, senescence) and **jumps back up** when a field is harvested and left as bare/stubble
soil (a much more temporally-stable scatterer). The *shape* of the coherence curve through
a season is therefore a strong, cloud-independent signal for crop phenology — no phase
unwrapping or displacement estimate is needed at all.

**Area of interest:** Flevoland, the Netherlands — a flat, homogeneous, intensively farmed
polder landscape that is a standard reference test site for Sentinel-1 agricultural
monitoring studies (e.g. WorldCereal, ESA's own S1 tutorials).

**What this notebook does:**
1. Discovers a Sentinel-1 burst over the AOI.
2. Builds a season-long, fixed short-baseline coherence time series with `sentinel1_sar_coherence`
   (VV and VH).
3. Applies a **custom UDF** along the time dimension that:
   - detects abrupt coherence *increases* (harvest / clear events) via a rolling z-score
     change-point test,
   - tracks the VH/VV coherence ratio as a canopy-density proxy through the season.
4. Visualises a harvest-date map and a per-field coherence/ratio curve.


In [ ]:
# --- Imports ---
import requests
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import openeo


In [ ]:
# --- Connect to the openEO back-end on CDSE ---
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()


## 1. Area of interest

A small block of arable parcels in Flevoland, NL.

In [ ]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [5.55, 52.55],
            [5.55, 52.60],
            [5.65, 52.60],
            [5.65, 52.55],
            [5.55, 52.55],
        ]
    ],
}


## 2. Find a Sentinel-1 burst covering the AOI

Same burst-discovery pattern as in the other notebooks — `sentinel1_sar_coherence` works
at burst level.

In [ ]:
def find_candidate_bursts(aoi_polygon, start, end, top=20):
    coords = aoi_polygon["coordinates"][0]
    wkt_coords = ", ".join(f"{lon} {lat}" for lon, lat in coords)
    footprint_wkt = f"POLYGON(({wkt_coords}))"

    filter_str = (
        f"OData.CSC.Intersects(area=geography'SRID=4326;{footprint_wkt}') "
        f"and ContentDate/Start gt {start}T00:00:00.000Z "
        f"and ContentDate/Start lt {end}T00:00:00.000Z "
        f"and PolarisationChannels eq 'VV&VH'"
    )
    url = (
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Bursts"
        f"?$filter={filter_str}&$top={top}&$orderby=ContentDate/Start asc"
    )
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json().get("value", [])


candidates = find_candidate_bursts(aoi, "2023-04-01", "2023-04-15")
for b in candidates:
    print(
        b.get("Id"), "|",
        "burst_id:", b.get("BurstId"),
        "swath:", b.get("SwathIdentifier"),
        "orbit:", b.get("RelativeOrbitNumber"),
        "direction:", b.get("OrbitDirection"),
    )


In [ ]:
BURST_ID = 234567          # <-- replace with a real burst_id for the AOI
SUB_SWATH = "IW1"          # <-- replace as needed

# Growing season: April (pre-emergence / bare soil) through September (post-harvest)
SEASON_START = "2023-04-01"
SEASON_END = "2023-09-30"
TEMPORAL_BASELINE_DAYS = 12   # fixed short baseline -> consecutive-pair coherence series


## 3. Build the seasonal coherence time series (VV and VH)

`temporal_extent` + `temporal_baseline` makes `sentinel1_sar_coherence` generate a whole
series of consecutive short-baseline pairs across the season automatically — this is a
datacube build, not a network inversion.

In [ ]:
def load_coherence_series(polarization):
    return connection.datacube_from_process(
        "sentinel1_sar_coherence",
        namespace=(
            "https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/"
            "algorithm_catalog/eurac/sentinel1_sar_coherence/openeo_udp/"
            "sentinel1_sar_coherence.json"
        ),
        **{
            "temporal_extent": [SEASON_START, SEASON_END],
            "temporal_baseline": TEMPORAL_BASELINE_DAYS,
            "burst_id": BURST_ID,
            "coherence_window_az": 2,
            "coherence_window_rg": 10,
            "polarization": polarization,
            "sub_swath": SUB_SWATH,
        },
    )


coherence_vv = load_coherence_series("VV").rename_labels(dimension="bands", target=["coherence_vv"])
coherence_vh = load_coherence_series("VH").rename_labels(dimension="bands", target=["coherence_vh"])

coherence_series = coherence_vv.merge_cubes(coherence_vh)


It's worth checking the actual temporal-dimension label and band names before writing
the UDF (`print(coherence_series.metadata)`), since UDP versions can differ; the UDF below
assumes a temporal dimension called `"t"` and bands `"coherence_vv"` / `"coherence_vh"`.

## 4. UDF: harvest/change-point detection + canopy-density proxy along time

For each pixel:
- take the VV coherence time series,
- compute the first difference through time,
- flag the date of the **largest positive, statistically-unusual jump** as a harvest/clear
  event (z-score of the first difference against the pixel's own seasonal variability),
- also output the seasonal-mean VH/VV coherence ratio as a canopy-density proxy.

This is purely a per-pixel time-series statistic — no unwrapping, no displacement, no
network inversion.


In [ ]:
harvest_detection_udf = """
import numpy as np
import xarray as xr

Z_THRESHOLD = 2.0  # how unusual a coherence jump must be to count as a harvest event


def apply_datacube(cube: xr.DataArray, context: dict) -> xr.DataArray:
    coh_vv = cube.sel(bands="coherence_vv")
    coh_vh = cube.sel(bands="coherence_vh")

    # --- harvest / change-point detection on VV coherence ---
    diff = coh_vv.diff("t")
    z = (diff - diff.mean("t")) / (diff.std("t") + 1e-6)

    # day-of-year of each time step (as a numeric proxy for the date, since the
    # UDF output has to stay numeric)
    doy = xr.DataArray(
        diff["t"].dt.dayofyear.values,
        dims="t",
        coords={"t": diff["t"]},
    )

    # mask out steps that are not a significant positive jump
    candidate_doy = doy.where(z > Z_THRESHOLD)

    # first significant jump per pixel = earliest harvest/clear event in the series
    harvest_doy = candidate_doy.min(dim="t", skipna=True)
    max_jump_z = z.max(dim="t", skipna=True)

    # --- seasonal canopy-density proxy: mean VH/VV coherence ratio ---
    ratio = (coh_vh / coh_vv.clip(min=1e-3)).mean(dim="t", skipna=True)

    result = xr.concat([harvest_doy, max_jump_z, ratio], dim="bands")
    result = result.assign_coords(bands=["harvest_doy", "max_jump_zscore", "vh_vv_ratio_mean"])
    return result
"""


In [ ]:
s1_vegetation_events = coherence_series.apply_dimension(
    code=harvest_detection_udf,
    runtime="Python",
    dimension="t",
)


## 5. Execute the batch job

In [ ]:
job = s1_vegetation_events.create_job(
    title="flevoland_harvest_detection",
    outputformat="netCDF",
)
job.start_and_wait()
job.get_results().download_files("flevoland_vegetation_events")


## 6. Plot the harvest-date map and a canopy-density (ratio) map

In [ ]:
result_ds = xr.load_dataset("flevoland_vegetation_events/openEO.nc")

fig, axes = plt.subplots(1, 2, figsize=(11, 5), dpi=100)

harvest_doy = result_ds["harvest_doy"] if "harvest_doy" in result_ds else (
    result_ds.to_array(dim="bands").sel(bands="harvest_doy")
)
ratio = result_ds["vh_vv_ratio_mean"] if "vh_vv_ratio_mean" in result_ds else (
    result_ds.to_array(dim="bands").sel(bands="vh_vv_ratio_mean")
)

harvest_doy.squeeze().plot.imshow(ax=axes[0], cmap="viridis", add_colorbar=True,
                                   cbar_kwargs={"label": "day of year"})
axes[0].set_title("Detected harvest / clear-event date")

ratio.squeeze().plot.imshow(ax=axes[1], cmap="YlGn_r", add_colorbar=True,
                             cbar_kwargs={"label": "mean VH/VV coherence ratio"})
axes[1].set_title("Seasonal canopy-density proxy")

for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


## 7. (Optional) per-field coherence curve

If you have a field boundary (e.g. from a parcel vector layer), `aggregate_spatial` the
`coherence_series` cube over that polygon before applying the UDF, and plot the raw VV/VH
coherence curve for a sanity check — useful for validating the change-point threshold before
scaling up to the full AOI.

## Notes & limitations

- `Z_THRESHOLD` and `TEMPORAL_BASELINE_DAYS` should be tuned per crop type / region; a
  coarser baseline reduces cost but blurs short events.
- This detects *a* significant coherence jump, not necessarily *the* harvest — co-occurring
  tillage, irrigation, or a rain event before the acquisition can also raise coherence.
  Cross-checking with a Sentinel-2 NDVI drop (via `merge_cubes` in the same openEO job) is a
  cheap way to add confidence without touching InSAR time-series inversion.
- No phase unwrapping, displacement, or network inversion was used anywhere in this workflow —
  everything here is pairwise coherence + a rolling window statistic in a UDF.
